# Contract protocol sketchboard

*Agents are **not** allowed to edit this notebook unless explicitly asked for.*

In [7]:

from dataclasses import dataclass
from steps import *

import portion as P
import interval_logic as il


@dataclass
class Contract:
    assumption: il.Formula | bool | str
    guarantee: il.Formula | bool | str

    def __post_init__(self) -> None:
        self.assumption = il.to_nnf(self.assumption).simplify()

        # Maintain saturated-form invariant.
        self.guarantee = il.to_nnf(il.Or((
            il.as_formula(self.guarantee),
            il.Not(self.assumption),
        ))).simplify()

    @staticmethod
    def _s(
        assumption: il.Formula | bool | str,
        guarantee: il.Formula | bool | str,
    ) -> str:
        return f"⟨ {il.as_formula(assumption)} ∣ {il.as_formula(guarantee)} ⟩"

    def __str__(self) -> str:
        return self._s(self.assumption, self.guarantee)

    @staticmethod
    def merge_s(self: "Contract", other: "Contract") -> str:
        return f"{self!s} ⊗ {other!s}"

    def merge(self, other: "Contract") -> "Contract":
        return Contract(
            il.And((self.assumption, other.assumption)),
            il.Or((
                il.And((self.guarantee, other.guarantee)),
                il.Not(il.And((self.assumption, other.assumption))),
            )),
        )

    @staticmethod
    def separate_s(self: "Contract", other: "Contract") -> str:
        return f"{self!s} ⦼ {other!s}"

    def separate(self, other: "Contract") -> "Contract":
        return Contract(
            il.Or((
                il.And((self.assumption, other.guarantee)),
                il.Not(il.And((self.guarantee, other.assumption))),
            )),
            il.And((self.guarantee, other.assumption)),
        )

    def to_nnf(self) -> "Contract":
        return Contract(
            il.to_nnf(self.assumption),
            il.to_nnf(self.guarantee),
        )

    def simplify(self) -> "Contract":
        return Contract(
            self.assumption.simplify(),
            self.guarantee.simplify(),
        )

    def replace(self, *fns) -> "Contract":
        return Contract(
            self.assumption.replace(*fns),
            self.guarantee.replace(*fns),
        )


## Request and offer objects

`Request.interval` is the allowed **start-time interval**.

`Offer.interval` is a concrete reserved charging interval.


In [8]:

@dataclass(frozen=True)
class Request:
    job: int
    duration: float
    interval: P.Interval

    @property
    def proposition(self) -> il.Formula:
        return il.Signal("j").equals(self.job)

    @property
    def contract(self) -> Contract:
        return Contract(
            True,
            il.Eventually(
                il.Always(self.proposition, self.duration),
                self.interval,
            ),
        )


@dataclass(frozen=True)
class Offer:
    job: int
    duration: float
    interval: P.Interval

    @property
    def contract(self) -> Contract:
        return Contract(
            True,
            il.Always(
                il.Signal("j").equals(self.job),
                self.interval,
            ),
        )


## Protocol transformations

Relative to target job `k`, the projection forgets which *other* job occupies the charger:

- `j = k` becomes `p`,
- `j = n`, `n != k`, becomes `¬p`.

This is a scheduler projection, not a generic logical equivalence.


In [9]:

p = il.Proposition("p")
j = il.Signal("j")


def project_job(job: int):
    target = il.NaturalLiteral(job)

    def replacer(psi):
        match psi:
            case il.Equal(left, right):
                if left == j:
                    return p if right == target else il.Not(p)
                if right == j:
                    return p if left == target else il.Not(p)

        return psi

    return replacer


def probe_strengthen(psi):
    # ∃_I H^d p -> ∀_I H^d p
    # The current AST keeps the resulting nested Always operators explicit.
    match psi:
        case il.Eventually(il.Always(phi, d), I):
            return il.Always(il.Always(phi, d), I).simplify()

    return psi


def weaken(psi):
    # Repair relaxation inherited from the old prototype:
    # ∀_I p -> ∃_I p
    match psi:
        case il.Always(phi, il.AbsoluteParameter(I)):
            return il.Eventually(phi, I)

    return psi


def overlap(psi):
    # ∀_I ¬p ∧ ∃_J p -> ∀_I ¬p ∧ ∃_(J-I) p
    match psi:
        case il.And((
            il.Always(phi, il.AbsoluteParameter(I)),
            il.Eventually(q, il.AbsoluteParameter(J)),
        )):
            base = q.body if isinstance(q, il.Always) else q
            if phi == il.Not(base):
                return il.And((
                    il.Always(phi, I),
                    il.Eventually(q, J - I),
                ))

        case il.And((
            il.Eventually(q, il.AbsoluteParameter(J)),
            il.Always(phi, il.AbsoluteParameter(I)),
        )):
            base = q.body if isinstance(q, il.Always) else q
            if phi == il.Not(base):
                return il.And((
                    il.Always(phi, I),
                    il.Eventually(q, J - I),
                ))

    return psi


def postprocess_repair(C_ms: Contract, job: int) -> Contract:
    C_post = Contract(
        True,
        C_ms.guarantee & C_ms.assumption,
    )

    C_post = C_post.replace(project_job(job)).simplify()
    C_post = C_post.replace(overlap).simplify()

    return C_post


## Offer generation

This notebook uses a simple fixed-duration grid policy after the contract operations expose a usable interval. The policy stays outside the contract algebra.


In [10]:

def grid_windows(
    interval: P.Interval,
    duration: float,
    *,
    step_size: float = 1.0,
) -> list[P.Interval]:
    windows = []

    for component in interval:
        start = float(component.lower)
        stop = float(component.upper)

        while start + duration <= stop:
            windows.append(P.closedopen(start, start + duration))
            start += step_size

    return windows


def generate_offers(
    job: int,
    interval: P.Interval,
    duration: float,
    *,
    step_size: float = 1.0,
) -> list[Offer]:
    return [
        Offer(job, duration, J)
        for J in grid_windows(
            interval,
            duration,
            step_size=step_size,
        )
    ]


def find_available_interval(
    expression: il.Formula,
) -> P.Interval | None:
    found = None

    def visit(psi):
        nonlocal found

        match psi:
            case il.Eventually(
                il.Always(phi, _),
                il.AbsoluteParameter(I),
            ):
                if phi == p:
                    found = I if found is None else found & I

        return psi

    expression.replace(visit)
    return found


# Case A — strengthened probe fails

Existing commitment:

\[
C^{cmt}
=
\langle \top \mid \forall_{[1,3)}(j=1) \rangle.
\]

Flexible request:

\[
C^{req}_2
=
\langle \top \mid \exists_{[2,4)} H^1(j=2) \rangle.
\]

For this toy notebook we explicitly follow the **failed-probe** branch so the repair algebra remains visible.


In [11]:

reset_step()

with step("Protocol — failed strengthened probe"):

    println("Commitments")
    printeq(
        C_cmt := Contract(True, "always_[1,3) j = 1"),
        C_cmt_p := C_cmt.replace(project_job(2)),
    )

    println("Request", top=1)
    req = Request(
        job=2,
        duration=1.0,
        interval=P.closedopen(2, 4),
    )
    printeq(
        C_req := req.contract,
        C_req_p := C_req.replace(project_job(req.job)),
    )

    with step("Probe strengthen"):
        printeq(
            C_req_p,
            C_str := C_req_p.replace(probe_strengthen),
        )

    with step("Merge strengthened"):
        printeq(
            Contract.merge_s(C_cmt_p, C_str),
            C_probe := C_cmt_p.merge(C_str),
        )

        println(
            "Probe branch for this example: EMPTY",
            "→ continue to repair availability",
            top=1,
        )

    with step("Repair availability"):

        println("Weaken")
        printeq(
            C_str,
            C_w := C_str.replace(weaken),
        )

        println("Separate", top=1)
        printeq(
            Contract.separate_s(C_w, C_cmt_p),
            C_s := C_w.separate(C_cmt_p),
        )

        println("Merge separated", top=1)
        printeq(
            Contract.merge_s(C_s, C_cmt_p),
            C_ms := C_s.merge(C_cmt_p),
        )

        println("Postprocess / project", top=1)
        printeq(
            Contract._s(True, C_ms.guarantee & C_ms.assumption),
            C_post := postprocess_repair(C_ms, req.job),
        )

    with step("Concretize"):
        available = find_available_interval(C_post.guarantee)

        println(f"Available region: {available}")

        offers = generate_offers(
            req.job,
            available,
            req.duration,
            step_size=1.0,
        )

        for n, offer in enumerate(offers, 1):
            println(
                f"Offer {n}",
                str(offer.contract),
            )


⏺ Step 1: Protocol — failed strengthened probe
│ Commitments
│ ⟨ ⊤ ∣ □_[1,3) j = 1 ⟩
│ = ⟨ ⊤ ∣ □_[1,3) ¬p ⟩ 
│ 
│ Request
│ ⟨ ⊤ ∣ ◇_[2,4) □_1 j = 2 ⟩
│ = ⟨ ⊤ ∣ ◇_[2,4) □_1 p ⟩  
├ ⏺ Step 1.1: Probe strengthen
│ │ ⟨ ⊤ ∣ ◇_[2,4) □_1 p ⟩  
│ │ = ⟨ ⊤ ∣ □_[2,4) □_1 p ⟩
├─╯ 
├ ⏺ Step 1.2: Merge strengthened
│ │ ⟨ ⊤ ∣ □_[1,3) ¬p ⟩ ⊗ ⟨ ⊤ ∣ □_[2,4) □_1 p ⟩
│ │ = ⟨ ⊤ ∣ □_[1,3) ¬p ∧ □_[2,4) □_1 p ⟩      
│ │ 
│ │ Probe branch for this example: EMPTY
│ │ → continue to repair availability   
├─╯ 
├ ⏺ Step 1.3: Repair availability
│ │ Weaken
│ │ ⟨ ⊤ ∣ □_[2,4) □_1 p ⟩  
│ │ = ⟨ ⊤ ∣ ◇_[2,4) □_1 p ⟩
│ │ 
│ │ Separate
│ │ ⟨ ⊤ ∣ ◇_[2,4) □_1 p ⟩ ⦼ ⟨ ⊤ ∣ □_[1,3) ¬p ⟩       
│ │ = ⟨ □_[1,3) ¬p ∨ □_[2,4) ◇_1 ¬p ∣ ◇_[2,4) □_1 p ⟩
│ │ 
│ │ Merge separated
│ │ ⟨ □_[1,3) ¬p ∨ □_[2,4) ◇_1 ¬p ∣ ◇_[2,4) □_1 p ⟩ ⊗ ⟨ ⊤ ∣ □_[1,3) ¬p ⟩                      
│ │ = ⟨ □_[1,3) ¬p ∨ □_[2,4) ◇_1 ¬p ∣ ◇_[1,3) p ∧ ◇_[2,4) □_1 p ∨ ◇_[2,4) □_1 p ∧ □_[1,3) ¬p ⟩
│ │ 
│ │ Postprocess / project
│ │ ⟨ ⊤ ∣ (◇_[1,3) p ∧ ◇_[2,4) □_1 p ∨

# Case B — strengthened probe succeeds

Here the request starts after the existing commitment:

\[
C^{req}_2
=
\langle \top \mid \exists_{[3,5)} H^1(j=2) \rangle.
\]

If the strengthened probe is nonempty, the repair path is skipped and concrete offers are generated directly from the original flexible request.


In [12]:

reset_step()

with step("Protocol — successful strengthened probe"):

    println("Commitments")
    printeq(
        C_cmt := Contract(True, "always_[1,3) j = 1"),
        C_cmt_p := C_cmt.replace(project_job(2)),
    )

    println("Request", top=1)
    req = Request(
        job=2,
        duration=1.0,
        interval=P.closedopen(3, 5),
    )
    printeq(
        C_req := req.contract,
        C_req_p := C_req.replace(project_job(req.job)),
    )

    with step("Probe strengthen"):
        printeq(
            C_req_p,
            C_str := C_req_p.replace(probe_strengthen),
        )

    with step("Merge strengthened"):
        printeq(
            Contract.merge_s(C_cmt_p, C_str),
            C_probe := C_cmt_p.merge(C_str),
        )

        println(
            "Probe branch for this example: NONEMPTY",
            "→ skip repair",
            top=1,
        )

    with step("Concretize original request"):
        offers = generate_offers(
            req.job,
            req.interval,
            req.duration,
            step_size=1.0,
        )

        for n, offer in enumerate(offers, 1):
            println(
                f"Offer {n}",
                str(offer.contract),
            )


⏺ Step 1: Protocol — successful strengthened probe
│ Commitments
│ ⟨ ⊤ ∣ □_[1,3) j = 1 ⟩
│ = ⟨ ⊤ ∣ □_[1,3) ¬p ⟩ 
│ 
│ Request
│ ⟨ ⊤ ∣ ◇_[3,5) □_1 j = 2 ⟩
│ = ⟨ ⊤ ∣ ◇_[3,5) □_1 p ⟩  
├ ⏺ Step 1.1: Probe strengthen
│ │ ⟨ ⊤ ∣ ◇_[3,5) □_1 p ⟩  
│ │ = ⟨ ⊤ ∣ □_[3,5) □_1 p ⟩
├─╯ 
├ ⏺ Step 1.2: Merge strengthened
│ │ ⟨ ⊤ ∣ □_[1,3) ¬p ⟩ ⊗ ⟨ ⊤ ∣ □_[3,5) □_1 p ⟩
│ │ = ⟨ ⊤ ∣ □_[1,3) ¬p ∧ □_[3,5) □_1 p ⟩      
│ │ 
│ │ Probe branch for this example: NONEMPTY
│ │ → skip repair                          
├─╯ 
├ ⏺ Step 1.3: Concretize original request
│ │ Offer 1              
│ │ ⟨ ⊤ ∣ □_[3,4) j = 2 ⟩
│ │ Offer 2              
│ │ ⟨ ⊤ ∣ □_[4,5) j = 2 ⟩
├─╯ 
╯


## Next iteration

Two pieces intentionally remain outside the contract algebra:

- **Nonemptiness / consistency:** replace the explicit branch labels with a scheduler satisfiability check over natural-number signals.
- **Offer policy:** `generate_offers` currently preserves duration on a fixed grid. Later policies can prioritize earliest start, maximum duration, partial duration/energy, etc.

The contract operations expose the temporal resource that survives; policy chooses the concrete commitment-ready `Always` contracts.
